<a href="https://colab.research.google.com/github/Sivasubramaniyan161004/Quantrix_Round2/blob/main/Quantrix_Round_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('/content/Round 02 - Dataset.csv')
#print(df.head())

In [ ]:
print("\n--- Checking for Null/Missing Values ---")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
if not missing_values.empty:
    print(missing_values)
else:
    print("No null or missing values found.")

print("\n--- Checking for Negative Values in Numeric Columns ---")
negative_values_found = False
for column in df.select_dtypes(include=['number']).columns:
    if (df[column] < 0).any():
        print(f"Column '{column}' contains negative values.")
        negative_values_found = True

if not negative_values_found:
    print("No negative values found in numeric columns.")


--- Checking for Null/Missing Values ---
children         4
country        488
agent        16340
company     112593
dtype: int64

--- Checking for Negative Values in Numeric Columns ---
Column 'adr' contains negative values.


In [ ]:
# Drop rows where 'adr' is negative
initial_rows = df.shape[0]
df_cleaned = df[df['adr'] >= 0]
rows_removed_negative = initial_rows - df_cleaned.shape[0]
print(f"Removed {rows_removed_negative} rows with negative 'adr' values.")

# Remove hard outliers from 'adr' using IQR method
Q1 = df_cleaned['adr'].quantile(0.25)
Q3 = df_cleaned['adr'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for hard outliers (typically Q1 - 3*IQR and Q3 + 3*IQR)
lower_bound = Q1 - (3 * IQR)
upper_bound = Q3 + (3 * IQR)

rows_before_outlier_removal = df_cleaned.shape[0]
df_cleaned = df_cleaned[(df_cleaned['adr'] >= lower_bound) & (df_cleaned['adr'] <= upper_bound)]
rows_removed_outliers = rows_before_outlier_removal - df_cleaned.shape[0]

print(f"Removed {rows_removed_outliers} hard outliers from 'adr' column.")
print(f"New DataFrame shape after cleaning: {df_cleaned.shape}")

df = df_cleaned.copy() # Update the main DataFrame 'df' with the cleaned data

Removed 1 rows with negative 'adr' values.
Removed 327 hard outliers from 'adr' column.
New DataFrame shape after cleaning: (119062, 32)


In [ ]:
df.groupby('hotel')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
hotel,,,
City Hotel,0.417383,79275,33088.0
Resort Hotel,0.277251,39787,11031.0


In [ ]:
df.groupby('deposit_type')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
deposit_type,,,
No Deposit,0.283670,104315,29591.0
Non Refund,0.993624,14585,14492.0
Refundable,0.222222,162,36.0


In [ ]:
df.groupby('market_segment')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
market_segment,,,
Aviation,0.219409,237,52.0
Complementary,0.130552,743,97.0
Corporate,0.187228,5293,991.0
Direct,0.153078,12510,1915.0
Groups,0.611434,19783,12096.0
Offline TA/TO,0.343232,24211,8310.0
Online TA,0.367002,56283,20656.0
Undefined,1.000000,2,2.0


In [ ]:
df.groupby('distribution_channel')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
distribution_channel,,,
Corporate,0.220674,6675,1473.0
Direct,0.174676,14524,2537.0
GDS,0.191710,193,37.0
TA/TO,0.410260,97665,40068.0
Undefined,0.800000,5,4.0


In [ ]:
df.groupby('customer_type')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
customer_type,,,
Contract,0.309617,4076,1262.0
Group,0.099130,575,57.0
Transient,0.407626,89327,36412.0
Transient-Party,0.254664,25084,6388.0


In [ ]:
df.groupby('previous_cancellations')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
previous_cancellations,,,
0,0.339122,112579,38178.0
1,0.944298,6050,5713.0
2,0.327586,116,38.0
3,0.307692,65,20.0
4,0.225806,31,7.0
5,0.105263,19,2.0
6,0.318182,22,7.0
11,0.285714,35,10.0
13,0.916667,12,11.0


In [ ]:
df.groupby('booking_changes')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
booking_changes,,,
0,0.408601,101084,41303.0
1,0.141964,12630,1793.0
2,0.201847,3790,765.0
3,0.153846,923,142.0
4,0.179625,373,67.0
5,0.173913,115,20.0
6,0.290323,62,18.0
7,0.096774,31,3.0
8,0.250000,16,4.0


In [ ]:
df['has_company'] = df['company'].notnull().astype(int)
df['has_agent'] = df['agent'].notnull().astype(int)

print("New columns 'has_company' and 'has_agent' created.")
print(df[['company', 'has_company', 'agent', 'has_agent']].head())

New columns 'has_company' and 'has_agent' created.
   company  has_company  agent  has_agent
0      NaN            0    NaN          0
1      NaN            0    NaN          0
2      NaN            0    NaN          0
3      NaN            0  304.0          1
4      NaN            0  240.0          1


In [ ]:
df.groupby('has_company')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
has_company,,,
0,0.382383,112267,42929.0
1,0.175129,6795,1190.0


In [ ]:
df.groupby('has_agent')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
has_agent,,,
0,0.247099,16285,4024.0
1,0.390116,102777,40095.0


In [ ]:
df.groupby('arrival_date_month')['is_canceled'].agg(['mean', 'count']).assign(
    impact_score=lambda x: x['mean'] * x['count']
)

,mean,count,impact_score
arrival_date_month,,,
April,0.408082,11086,4524.0
August,0.377912,13691,5174.0
December,0.350934,6742,2366.0
February,0.334160,8068,2696.0
January,0.304773,5929,1807.0
July,0.374811,12585,4717.0
June,0.414601,10931,4532.0
March,0.321487,9792,3148.0
May,0.396604,11780,4672.0


In [ ]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['revenue_lost'] = df['adr'] * df['total_nights']

In [ ]:
def revenue_loss_summary(df, group_col):
    total_potential = df.groupby(group_col)['revenue_lost'].sum()
    lost = df[df['is_canceled']==1].groupby(group_col)['revenue_lost'].sum()

    summary = pd.DataFrame({
        'total_potential_revenue': total_potential,
        'revenue_lost': lost,
        'loss_pct': (lost / total_potential * 100).round(2)
    }).sort_values('revenue_lost', ascending=False)

    return summary

In [ ]:
categories = ['hotel', 'deposit_type', 'market_segment', 'distribution_channel',
              'customer_type', 'has_company', 'has_agent', 'arrival_date_month']

results = {}
for col in categories:
    results[col] = revenue_loss_summary(df, col)
    print(f"\n--- {col} ---")
    print(results[col])


--- hotel ---
              total_potential_revenue  revenue_lost  loss_pct
hotel                                                        
City Hotel                25218077.01   10866063.81     43.09
Resort Hotel              17048700.77    5697319.23     33.42

--- deposit_type ---
              total_potential_revenue  revenue_lost  loss_pct
deposit_type                                                 
No Deposit                38602075.50   12944777.84     33.53
Non Refund                 3615716.90    3598607.03     99.53
Refundable                   48985.38      19998.17     40.82

--- market_segment ---
                total_potential_revenue  revenue_lost  loss_pct
market_segment                                                 
Online TA                   23670727.99   10100105.73     42.67
Groups                       4652526.54    2799863.98     60.18
Offline TA/TO                8136440.73    2486958.15     30.57
Direct                       4941348.38     963806.12     19.

In [ ]:
# Bucket into a clean categorical flag
df['prev_cancel_flag'] = df['previous_cancellations'].apply(lambda x: '1+' if x > 0 else '0')

# Reuse your existing function
revenue_loss_summary(df, 'prev_cancel_flag')

,total_potential_revenue,revenue_lost,loss_pct
prev_cancel_flag,,,
0,40857318.44,15221639.16,37.26
1+,1409459.34,1341743.88,95.20


In [ ]:
# Bucket into a clean categorical flag
df['booking_changes_flag'] = df['booking_changes'].apply(lambda x: '1+' if x > 0 else '0')

# Reuse your existing function
revenue_loss_summary(df, 'booking_changes_flag')

,total_potential_revenue,revenue_lost,loss_pct
booking_changes_flag,,,
0,35047350.19,15149843.63,43.23
1+,7219427.59,1413539.41,19.58
